In [23]:
print("Notebook läuft")

Notebook läuft


# Beyond Words – Audio Feature Extraction

## Forschungsfrage 1
Wie können emotionale und prosodische Merkmale aus Audiodaten extrahiert und strukturiert repräsentiert werden?

Dieses Notebook implementiert schrittweise eine Pipeline zur Extraktion von Merkmalen aus Sprachaufnahmen.

Iteratives Vorgehen:
1. Transcript erzeugen
2. Sprechtempo berechnen
3. Lautstärke bestimmen
4. Emotion erkennen
5. Strukturierte Repräsentation als JSON

Warum brauchen wir librosa?

librosa erlaubt uns:

Dauer der Aufnahme berechnen
Lautstärke analysieren
Pausen erkennen
weitere Audio Features extrahieren

Das ist zentral für deine Forschungsfrage:

prosodische Merkmale aus Audio extrahieren

In [ ]:
import os
import json
import librosa

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY fehlt. Bitte in der .env-Datei setzen.")

client = OpenAI(api_key=api_key)

audio_path = "../audio/gestresst.wav"

print("Setup erfolgreich")

In [6]:
try:
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="gpt-4o-mini-transcribe",
            file=audio_file
        )
except Exception as exc:
    raise RuntimeError(
        "Transkription fehlgeschlagen. Bitte OPENAI_API_KEY in .env pruefen und die Zelle erneut ausfuehren."
    ) from exc

text = transcript.text.strip()

print("Transcript:")
print(text)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************Go0A. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [4]:
duration_seconds = librosa.get_duration(path=audio_path)

print("Dauer:", duration_seconds)

c:\Users\jdonati\OneDrive - Axept Business Software AG\Desktop\Jaden Donati\Privat\BA\Code_BA_V2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dauer: 7.04


Nächste Notebook-Zelle

Jetzt berechnen wir die Wörter:

In [ ]:
if "text" not in globals() or not text:
    raise RuntimeError("Bitte zuerst die Transkriptionszelle erfolgreich ausfuehren.")

word_count = len(text.split())

print("Anzahl Wörter:", word_count)

Anzahl Wörter: 15


word_count = len(text.split())

print("Anzahl Wörter:", word_count)

In [5]:
if "word_count" not in globals():
    raise RuntimeError("Bitte zuerst die Wortzaehlungs-Zelle ausfuehren.")

words_per_minute = (word_count / duration_seconds) * 60

print("Wörter pro Minute:", words_per_minute)

NameError: name 'word_count' is not defined

Speech Rate Kategorien definieren 

In [4]:
if "words_per_minute" not in globals():
    raise RuntimeError("Bitte zuerst die Zelle fuer Woerter pro Minute ausfuehren.")

if words_per_minute < 110:
    speech_speed = "slow"

elif words_per_minute < 160:
    speech_speed = "normal"

else:
    speech_speed = "fast"

print("Speech speed category:", speech_speed)

NameError: name 'words_per_minute' is not defined

Jetzt erstellen wir daraus eine JSON

In [ ]:
result = {
    "audio_file": "gestresst.wav",
    "transcript": text,
    "audio_duration_seconds": round(duration_seconds, 2),
    "word_count": word_count,
    "prosody": {
        "speech_rate": {
            "words_per_minute": round(words_per_minute, 2),
            "category": speech_speed
        }
    }
}

print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "audio_file": "gestresst.wav",
  "transcript": "I am really stressed today because I have many deadlines and not much time left.",
  "audio_duration_seconds": 7.04,
  "word_count": 15,
  "prosody": {
    "speech_rate": {
      "words_per_minute": 127.84,
      "category": "normal"
    }
  }
}


In [26]:
with open("../audio/gestresst_analysis.json", "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print("JSON gespeichert")

JSON gespeichert
